# Streamflow Anomaly Hex-Grid
Given our recent PNW atmospheric river I thought I would try playing with hexagon visualizaiton of streamflow surge for the 30 Day Map Challenge. Instead of drawing standard blue lines that get cluttered at regional scales, this map aggregates NOAA National Water Model (NWM) channel routing streamflow into Uber H3 hexagons. For each hexagon, flow at every time step is divided by its flow at the start of the window (Nov 9, 00:00 UTC). Each hexagon then acts as a hydrological "pixel" showing where rivers are surging (above 1x) or receding (below 1x) during the event.

<iframe src="map.html" width="100%" height="600" style="border:0; border-radius:8px;" loading="lazy" title="Animated map of Pacific Northwest streamflow surge in H3 hexagons"></iframe>

*Press play or drag the slider to step through Nov 9–14, 2026. Height and color both show the flow multiplier relative to Nov 9, 00:00 UTC.*

# Tech Stack
• Data Access: xarray + fsspec + s3fs (with zarr and h5netcdf) to stream NWM data directly from AWS S3 without downloading large archives.
• Spatial Indexing: h3 (v4+) and pandas. Hexagon polygons are built directly from H3 boundaries, so no geopandas or shapely is needed.
• Web Display: a hand-written MapLibre GL JS page (no Python mapping library) that animates 3D fill-extrusions.

# Step-By-Step Python Workflow

Step 1: Build a Spatial Hex-Lookup (reach coordinates → H3)

• Why: Operational NWM streamflow files only contain river reach IDs (feature_id), not coordinates.
• What the script does: It reads each reach's latitude and longitude from the NWM v3.0 retrospective Zarr store (`chrtout.zarr`), filters to the Pacific Northwest (42°N–49°N, -124.8°W to -116.5°W), assigns each reach to an H3 Resolution 6 hexagon (~36 km²), and caches the lookup as pnw_routelink.parquet so later runs are fast. (The RouteLink_CONUS.nc file is not hosted in this bucket, so the Zarr store is used for coordinates only.)

Step 2: Stream Recent Atmospheric River Data from AWS S3

• Why: NOAA publishes hourly NWM analysis_assim (assimilated streamflow) files on AWS Open Data (s3://noaa-nwm-pds), but only for a rolling window of about four weeks.
• What the script does: It streams 6-hourly snapshots for Nov 9–14, 2026 (24 frames) with fsspec and xarray, keeping only the PNW reaches in memory instead of downloading the full national files.

Step 3: Aggregate & Normalize into Surge Multipliers

• Why: Raw flow in m³/s varies enormously between small coastal streams and the Columbia River, so values are normalized per hexagon. Some reaches also behave unnaturally (for example regulated dam releases that switch on and off), and these would distort the map.
• What the script does: First it skips reaches whose flow jumps by more than 10x (up or down) between consecutive steps more than twice, since a real storm hydrograph rises and falls about once. Each hexagon is then represented by its largest remaining reach (the one with the highest mean flow over the window), so a cell holding only a short stretch of a big river is not swamped by many small tributaries; a hexagon with no usable reach is dropped. Hexagons whose reach has a baseline under 1 m³/s are also dropped. Every time step is divided by that reach's Nov 9 00:00 UTC flow, with the baseline floored at 5 m³/s and the result capped at 10x: 1.0 means unchanged, 5.0 means five times higher. Because of the floor, streams with a baseline under 5 m³/s start below 1x, and the multiplier is relative to each hexagon's own baseline, so a creek that doubles looks like the Columbia doubling only when both baselines are above the floor.

Step 4: Export GeoJSON & JS Data File

• Why: A static data file plus a single HTML page can be hosted on GitHub Pages with no server-side code.
• What the script does: It writes the hexagon polygons and their per-frame multipliers to a minified pnw_hex_surge.geojson for reuse, and the same data to pnw_hex_data.js (`window.HEX_DATA = ...`). map.html loads the .js file with a script tag instead of `fetch`, so the map works from a server, GitHub Pages, or a double-click.

Step 5: GPU-Accelerated MapLibre Animation

• How the web map works: The whole time series is stored in each hexagon's `surge` array. Because MapLibre turns array properties into strings, the page first copies each frame's value into its own property (`s0`, `s1`, ...). Each frame, it updates fill-extrusion-height and fill-extrusion-color with the expression `["get", "s" + frameIndex]`, so the geometry is never reloaded. Height and color both show the flow multiplier; the on-map legend explains this.


In [20]:
import os
import json
import h3
import fsspec
import numpy as np
import xarray as xr
import pandas as pd

BASELINE_FLOOR_CMS = 5.0  # denominator floor (m³/s) so tiny baselines don't inflate ratios
SURGE_CAP = 10.0          # multipliers are capped here (matches the "10x+" legend)
JUMP_FACTOR = 10.0        # a step-to-step change larger than this (up or down) is a "jump"
MAX_JUMPS = 2             # reaches with more jumps than this look regulated/erratic and are skipped

# ==========================================
# 1. FETCH & CACHE PNW COORDINATE LOOKUP
# ==========================================
ROUTELINK_CACHE = "pnw_routelink.parquet"

if os.path.exists(ROUTELINK_CACHE):
    meta = pd.read_parquet(ROUTELINK_CACHE)
else:
    print("Reading NWM reach coordinates from the retrospective Zarr (one-time)...")
    # RouteLink_CONUS.nc is not hosted in this bucket; the Zarr store carries
    # per-reach latitude/longitude keyed by feature_id.
    zarr_url = "s3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/chrtout.zarr"
    ds_rl = xr.open_zarr(zarr_url, storage_options={"anon": True})
    meta = pd.DataFrame({
        "feature_id": ds_rl["feature_id"].values,
        "lat": ds_rl["latitude"].values,
        "lon": ds_rl["longitude"].values,
    })

    # Filter to Pacific Northwest bounding box (OR, WA, parts of ID)
    meta = meta[
        (meta["lat"].between(42.0, 49.0)) &
        (meta["lon"].between(-124.8, -116.5))
    ].copy()

    # Assign H3 Resolution 6 hexagons (~36 km² cells)
    meta["h3_cell"] = [
        h3.latlng_to_cell(la, lo, 6) for la, lo in zip(meta["lat"], meta["lon"])
    ]
    meta.to_parquet(ROUTELINK_CACHE, index=False)

pnw_ids = set(meta["feature_id"])
print(f"Tracking {len(pnw_ids):,} PNW river reaches across {meta['h3_cell'].nunique():,} hexagons.")

# ==========================================
# 2. STREAM THIS WEEK'S NWM DATA (NOV 9–14, 2026)
# ==========================================
dates = pd.date_range("2026-11-09 00:00", "2026-11-14 18:00", freq="6h")
ts_dict = {}

for dt in dates:
    d_str = dt.strftime("%Y%m%d")
    h_str = dt.strftime("%H")
    url = f"s3://noaa-nwm-pds/nwm.{d_str}/analysis_assim/nwm.t{h_str}z.analysis_assim.channel_rt.tm00.conus.nc"

    try:
        with fsspec.open(url, anon=True) as f:
            ds = xr.open_dataset(f, engine="h5netcdf")
            mask = pd.Series(ds["feature_id"].values).isin(pnw_ids).values
            ids = ds["feature_id"].values[mask]
            flow = ds["streamflow"].values[mask]
            ts_dict[dt] = pd.Series(flow, index=ids)
    except Exception as e:
        print(f"Skipped {dt.strftime('%Y-%m-%d %H:%M UTC')}: {e}")
print(f"Loaded {len(ts_dict)} of {len(dates)} time steps.")

# ==========================================
# 3. AGGREGATE & COMPUTE SURGE MULTIPLIERS
# ==========================================
q_df = pd.DataFrame(ts_dict)
q_df["h3_cell"] = meta.set_index("feature_id")["h3_cell"]
flow_cols = [c for c in q_df.columns if c != "h3_cell"]

# Flag reaches that jump up and down sharply between 6-hourly steps. A real storm
# hydrograph rises and falls about once; repeated on/off swings point to regulated
# releases (e.g. hydropower dams) or erratic model output, not natural flow.
# Flows are floored at 1 m³/s so trickles near zero don't count as jumps.
arr = q_df[flow_cols].clip(lower=1.0).to_numpy()
step = arr[:, 1:] / arr[:, :-1]
q_df["_jumps"] = ((step > JUMP_FACTOR) | (step < 1 / JUMP_FACTOR)).sum(axis=1)
erratic = q_df["_jumps"] > MAX_JUMPS
print(f"Skipping {int(erratic.sum()):,} erratic reaches (> {MAX_JUMPS} jumps of {JUMP_FACTOR:g}x).")

# Use the largest remaining reach in each hexagon (highest mean flow) as its
# representative time series. A cell-wide percentile is dominated by many tiny
# tributaries and drops cells that hold only a short stretch of a big river
# (e.g. the Columbia). Hexagons with no non-erratic reach drop out.
q_df["_mean"] = q_df[flow_cols].mean(axis=1)
main_reach = q_df[~erratic].sort_values("_mean", ascending=False).groupby("h3_cell").head(1)
hex_ts = main_reach.set_index("h3_cell")[flow_cols]

# Filter out tiny headwater/dry cells (baseline < 1 m³/s at the first time step)
baseline = hex_ts.iloc[:, 0]
hex_ts = hex_ts[baseline > 1.0].copy()
baseline = hex_ts.iloc[:, 0]
peak_flow = hex_ts.max(axis=1)

# Surge ratio relative to the Nov 9 baseline (1.0 = normal, 5.0 = 5x flood surge).
# The denominator is floored and the result capped so that tiny baselines
# cannot dominate the scale.
denominator = baseline.clip(lower=BASELINE_FLOOR_CMS)
surge_ratio = hex_ts.div(denominator, axis=0).clip(upper=SURGE_CAP).round(2)
timestamps = [t.strftime("%b %d, %H:%M UTC") for t in surge_ratio.columns]
print(f"Kept {len(hex_ts):,} hexagons.")

# ==========================================
# 4. EXPORT GEOJSON & JS DATA FILE
# ==========================================
features = []
for cell_id, row in surge_ratio.iterrows():
    boundary = h3.cell_to_boundary(cell_id)
    coords = [[round(lon, 4), round(lat, 4)] for lat, lon in boundary]
    coords.append(coords[0])  # Close polygon ring

    features.append({
        "type": "Feature",
        "geometry": {"type": "Polygon", "coordinates": [coords]},
        "properties": {
            "h3": cell_id,
            "base_cms": round(float(baseline.loc[cell_id]), 1),
            "peak_cms": round(float(peak_flow.loc[cell_id]), 1),
            "max_surge": round(float(row.max()), 2),
            "surge": row.values.tolist(),
        }
    })

geojson_out = {
    "type": "FeatureCollection",
    "timestamps": timestamps,
    "features": features
}
payload = json.dumps(geojson_out, separators=(",", ":"))

# Plain GeoJSON for reuse in other tools
with open("pnw_hex_surge.geojson", "w") as f:
    f.write(payload)
print(f"Saved pnw_hex_surge.geojson ({len(features)} hexagons, {len(timestamps)} frames).")

# Same data as a script that map.html loads with a <script> tag. Unlike fetch(),
# this works from a web server, GitHub Pages, and when opened from file://.
with open("pnw_hex_data.js", "w", encoding="utf-8") as f:
    f.write(f"window.HEX_DATA = {payload};\n")
print("Saved pnw_hex_data.js (loaded by map.html).")


Tracking 139,004 PNW river reaches across 12,840 hexagons.
Loaded 24 of 24 time steps.
Skipping 4 erratic reaches (> 2 jumps of 10x).
Kept 3,187 hexagons.
Saved pnw_hex_surge.geojson (3187 hexagons, 24 frames).
Saved pnw_hex_data.js (loaded by map.html).


# Caveats

This map shows **modeled** National Water Model (NWM) analysis flows, not observed river conditions, and the multiplier can overstate how dramatic an event was.

• **Low-baseline effect:** every hexagon is compared with its own flow at Nov 9, 00:00 UTC, when many rivers were near their autumn minimum. A small stream at 2–13 m³/s that rises after modest rain produces a large multiplier, so height and color flag a big relative change, not a big or dangerous flow. The baseline floor (5 m³/s) and the 10x cap soften this but do not remove it. Big, steady rivers such as the Rogue barely register.

• **NWM overshoot in SW Oregon:** the hexagons along the southern Oregon coast that hit 10x are partly a model effect. Comparing the nearest NWM reach with USGS gauges (provisional data) for Nov 9–14 gave:

| River | USGS, first → peak (m³/s) | NWM, first → peak (m³/s) |
|---|---|---|
| Chetco near Brookings | 3.8 → 85 | 13.5 → 386 |
| Sixes at Sixes | 2.0 → 32 | 3.5 → 159 |
| South Fork Coquille at Powers | 0.5 → 19 | 1.2 → 42 |

The gauges did record a real rise on Nov 13–14, but it was much smaller in absolute terms than the model suggests (the Chetco peaked near 3,000 cfs), and the model's starting flows are also too high. The comparison used the nearest reaches and 6-hourly model snapshots, so it is approximate.

• **Other simplifications:** each hexagon is represented by its single largest river reach, and reaches that switch on and off sharply (such as regulated dam releases) are skipped. Check an observed source such as the USGS National Water Dashboard for actual conditions.


Part 2: Animated MapLibre HTML Map (map.html)

Instead of reloading geometries on every frame, the HTML page keeps each hexagon's multipliers in its properties (flattened to `s0`, `s1`, ... on load) and updates MapLibre's fill-extrusion-height and fill-extrusion-color expressions with `["get", "s" + frameIndex]`. It includes a play/pause button, a time slider, and a legend, and the layout adapts to narrow screens and small iframes. The data comes from `pnw_hex_data.js`, loaded with a script tag rather than `fetch`, so the page works on a web server, on GitHub Pages, and when opened directly from disk (it still needs an internet connection for MapLibre and the basemap).

The collapsed section below shows the current map.html from this folder, so it always matches the file on disk.


In [16]:
#| echo: false
import html
from IPython.display import HTML

with open("map.html", "r", encoding="utf-8") as f:
    map_src = f.read()

HTML(
    "<details><summary>Show the map.html source</summary>"
    f"<pre><code>{html.escape(map_src)}</code></pre></details>"
)